In [2]:
import logging


In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler("pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [4]:
logger.info("Run command: pip install requests")
%pip install -q requests

2026-04-30 20:11:59,433 - INFO - Run command: pip install requests



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import requests

logging.info("Fetching weather data.")
url = "https://api.open-meteo.com/v1/forecast?latitude=47.9&longitude=106.9&current_weather=true"

response = requests.get(url)
print(response.status_code)
response.json()

2026-04-30 20:12:03,020 - INFO - Fetching weather data.


200


{'latitude': 47.90861,
 'longitude': 106.86567,
 'generationtime_ms': 0.07963180541992188,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 1287.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-04-30T12:00',
  'interval': 900,
  'temperature': 7.1,
  'windspeed': 12.0,
  'winddirection': 351,
  'is_day': 1,
  'weathercode': 2}}

In [6]:
import pandas as pd
from datetime import datetime

logging.info("Creating dataframe.")
data = response.json()["current_weather"]
data["fetched_at"] = datetime.utcnow()

df = pd.DataFrame([data])
df.head(3)

2026-04-30 20:12:18,842 - INFO - Creating dataframe.
/var/folders/hv/wjvkzpmj6fv44k0l243tmj_h0000gn/T/ipykernel_36379/169850088.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  data["fetched_at"] = datetime.utcnow()


,time,interval,temperature,windspeed,winddirection,is_day,weathercode,fetched_at
0,2026-04-30T12:00,900,7.1,12.0,351,1,2,2026-04-30 12:12:18.842896


In [7]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import os
from dotenv import load_dotenv

load_dotenv("../.env")

DB_URL = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=os.getenv("SUPABASE_PASSWORD"),
    host="db.vqssaigndessrttplwib.supabase.co",
    port=5432,
    database="postgres"
)

try:
    engine = create_engine(DB_URL)
    df.to_sql("weather_raw", engine, if_exists="append", index=False)
    logging.info("Database created successfully.")
except Exception as e:
    logging.info(f"Error: {e}")

2026-04-30 20:12:28,221 - INFO - Database created successfully.


In [8]:
logging.info("Run command: pip install schedule")
%pip install -q schedule

2026-04-30 20:15:19,125 - INFO - Run command: pip install schedule



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
